In [1]:
#imports

!pip install numpy pandas matplotlib scikit-learn ucimlrepo seaborn
!pip install ucimlrepo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from ucimlrepo import fetch_ucirepo
from sklearn import preprocessing
from sklearn.preprocessing import OneHotEncoder
import seaborn as sns


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
df = pd.read_csv(   "https://raw.githubusercontent.com/guipsamora/pandas_exercises/master/04_Apply/Students_Alcohol_Consumption/student-mat.csv", sep=None, engine="python")

In [3]:
student_performance = df
X = student_performance.drop(["G3", "G2", "G1"], axis=1)
y = student_performance[["G3", "G2", "G1"]]
y_G3 = y["G3"]


In [4]:
categorical_features_nominal = X.select_dtypes(include=["str"]).columns
categorical_features_ordinal = X.select_dtypes(include=["int64", "float64"]).columns
print(f"Categorical features nominal: {categorical_features_nominal}")
print(f"Categorical features ordinal: {categorical_features_ordinal}")


Categorical features nominal: Index(['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob',
       'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities',
       'nursery', 'higher', 'internet', 'romantic'],
      dtype='str')
Categorical features ordinal: Index(['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel',
       'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences'],
      dtype='str')


In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y_G3, test_size=0.2, random_state=42)

In [6]:
one_hot = OneHotEncoder(sparse_output=False, handle_unknown='ignore') 
one_hot_encoded_train = one_hot.fit_transform(X_train[categorical_features_nominal]) #only transforming the training set, we will use the same encoder to transform the test set later on
X_train_nominal = pd.DataFrame(one_hot_encoded_train, columns=one_hot.get_feature_names_out(categorical_features_nominal), index=X_train.index) #creating a new dataframe for the one hot encoded features, and keeping the same index as the original dataframe

one_hot_encoded_test = one_hot.transform(X_test[categorical_features_nominal])
X_test_nominal = pd.DataFrame(one_hot_encoded_test, columns=one_hot.get_feature_names_out(categorical_features_nominal), index=X_test.index) #creating a new dataframe for the one hot encoded features, and keeping the same index as the original dataframe
    
#add the new df in places of the old ones for both training and test sets
X_train_encoded = pd.concat(
    [X_train.drop(columns=categorical_features_nominal), X_train_nominal],
    axis=1
)

X_test_encoded = pd.concat(
    [X_test.drop(columns=categorical_features_nominal), X_test_nominal],
    axis=1
)

In [8]:
lr = sklearn.linear_model.LinearRegression()
lr.fit(X_train_encoded, y_train)
y_pred_lr = lr.predict(X_test_encoded)

#metrics for linear regression
from sklearn.metrics import mean_absolute_error, r2_score
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)
print(f"Linear Regression MAE: {mae_lr}")
print(f"Linear Regression R2 Score: {r2_lr}")

Linear Regression MAE: 3.3952609258019226
Linear Regression R2 Score: 0.1414924741119573
